<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 1. Aspectos generales

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast
from pyspark.sql.types import DateType

In [2]:
# Inicializar SparkSession con configuracion de memoria
spark = SparkSession.builder \
    .appName("HYM_Bronze_to_Silver") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.memory.fraction", "0.8") \
    .getOrCreate()

spark.sparkContext.setCheckpointDir("/tmp/spark_checkpoint")
spark

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 2. Rutas de capas

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Define las tres rutas de la arquitectura medallion.
Los datos de entrada se leen siempre desde **bronce**;
el resultado se escribe en **silver**.

In [3]:
BRONCE = "./bronce"
SILVER = "./silver"
GOLD   = "./gold"

import os
os.makedirs(SILVER, exist_ok=True)
os.makedirs(GOLD,   exist_ok=True)

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 3. Lectura desde capa bronce

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Lectura de los tres archivos fuente:

- `articles.csv`: catalogo de productos.
- `customers.csv`: datos demograficos de clientes.
- `transactions.parquet`: historial de transacciones; se castea `t_dat` a `DateType`
  y se renombra a `Fecha` para consistencia con los notebooks siguientes.

In [4]:
%%time
articles_ = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BRONCE}/articles.csv")
)

transactions_ = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .parquet(f"{BRONCE}/transactions.parquet")
    .withColumn("t_dat", col("t_dat").cast(DateType()))
    .withColumnRenamed("t_dat", "Fecha")
)

customers_ = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BRONCE}/customers.csv")
)

CPU times: user 2.24 ms, sys: 7 ms, total: 9.24 ms
Wall time: 5.12 s


<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 4. Inspeccion de datos

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Revision rapida del esquema y primeras filas de cada fuente antes de realizar el join.

In [5]:
print("--- articles ---")
print(f"columnas: {len(articles_.columns)}  |  schema:")
articles_.printSchema()

print("--- customers ---")
customers_.printSchema()

print("--- transactions ---")
transactions_.printSchema()

--- articles ---
columnas: 25  |  schema:
root
 |-- article_id: integer (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: integer (nullable = true)
 |-- 

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 5. Join y escritura en capa silver

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Une las tres fuentes por `article_id` y `customer_id` (inner join).

Se aplica `broadcast` a `articles_` y `customers_` porque son tablas de dimension
(mucho mas pequenas que `transactions_`), lo que evita un shuffle completo y
reduce significativamente el tiempo de ejecucion.

El DataFrame resultante se escribe en `silver/merge_pyspark` en formato Parquet.

In [6]:
%%time
merged_df = (
    transactions_
    .join(broadcast(articles_), "article_id", "inner")
    .join(broadcast(customers_), "customer_id", "inner")
)

merged_df.write.mode("overwrite").parquet(f"{SILVER}/merge_pyspark")
print("Escritura en silver completada")

Escritura en silver completada
CPU times: user 86.3 ms, sys: 72.8 ms, total: 159 ms
Wall time: 4min 37s


<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 6. Verificacion

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Lee el archivo recien escrito desde silver y valida el conteo de registros y el esquema.

In [7]:
df_silver = spark.read.parquet(f"{SILVER}/merge_pyspark")
print(f"Total registros en silver/merge_pyspark: {df_silver.count():,}")
df_silver.printSchema()

Total registros en silver/merge_pyspark: 31,788,324
root
 |-- customer_id: string (nullable = true)
 |-- article_id: long (nullable = true)
 |-- Fecha: date (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: long (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department

In [8]:
spark.stop()